In [10]:
from google.colab import drive
from pathlib import Path
import os
import shutil
import pyarrow.parquet as pq

import pandas as pd
try:
    drive.flush_and_unmount()
except Exception:
    pass

if os.path.exists("/content/drive"):
    shutil.rmtree("/content/drive", ignore_errors=True)

drive.mount(
    "/content/drive",
    force_remount=True,
    timeout_ms=300000,
)

MY_DRIVE = Path("/content/drive/MyDrive")

if not MY_DRIVE.exists():
    raise RuntimeError("MyDrive is not available after mounting.")

DATA_DIR = MY_DRIVE / "Language Detection"

if not DATA_DIR.exists():
    raise FileNotFoundError(
        f"Folder not found: {DATA_DIR}"
    )

PARQUET_PATH = (
    DATA_DIR
    / "sessions_lang_transcript_2026-08-23_2026-08-24.parquet"
)

if not PARQUET_PATH.is_file():
    available_files = [
        path.name
        for path in DATA_DIR.iterdir()
        if path.is_file()
    ]

    raise FileNotFoundError(
        f"Parquet file not found: {PARQUET_PATH}\n"
        f"Available files: {available_files}"
    )

parquet_file = pq.ParquetFile(PARQUET_PATH)
metadata = parquet_file.metadata

print("File path   :", PARQUET_PATH)
print("File size   :", f"{PARQUET_PATH.stat().st_size / (1024**2):.2f} MB")
print("Rows        :", f"{metadata.num_rows:,}")
print("Row groups  :", metadata.num_row_groups)
print("Columns     :", metadata.num_columns)
print()
print(parquet_file.schema)

Drive not mounted, so nothing to flush and unmount.
Mounted at /content/drive
File path   : /content/drive/MyDrive/Language Detection/sessions_lang_transcript_2026-08-23_2026-08-24.parquet
File size   : 469.49 MB
Rows        : 3,469
Row groups  : 1
Columns     : 12

required group field_id=-1 schema {
  optional int64 field_id=-1 gamesession_id;
  optional int64 field_id=-1 user_id;
  optional binary field_id=-1 game_name (String);
  optional binary field_id=-1 url (String);
  optional binary field_id=-1 model_type (String);
  optional binary field_id=-1 created_at (String);
  optional binary field_id=-1 lang_detected (String);
  optional double field_id=-1 lang_probability;
  optional group field_id=-1 transcript_segments (List) {
    repeated group field_id=-1 list {
      optional group field_id=-1 element {
        optional binary field_id=-1 text (String);
        optional group field_id=-1 timestamp (List) {
          repeated group field_id=-1 list {
            optional double 

In [11]:
con = duckdb.connect()
pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", None)
pd.set_option("display.width", None)
preview_df = con.execute(
    f"""
    SELECT *
    FROM read_parquet('{PARQUET_PATH.as_posix()}')
    LIMIT 1
    """
).df()

display(preview_df)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

gamesession_id  user_id           game_name  \
0       141231477   687197  Escape from Tarkov   

                                       url model_type           created_at  \
0  https://www.twitch.tv/videos/2853161891      gen10  2026-08-23 00:00:37   

  lang_detected  lang_probability  \
0            de            0.9619   

                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                      

In [12]:
schema_df = con.execute(
    f"""
    DESCRIBE
    SELECT *
    FROM read_parquet('{PARQUET_PATH.as_posix()}')
    """
).df()

display(schema_df)

,column_name,column_type,null,key,default,extra
0,gamesession_id,BIGINT,YES,None,None,None
1,user_id,BIGINT,YES,None,None,None
2,game_name,VARCHAR,YES,None,None,None
3,url,VARCHAR,YES,None,None,None
4,model_type,VARCHAR,YES,None,None,None
5,created_at,VARCHAR,YES,None,None,None
6,lang_detected,VARCHAR,YES,None,None,None
7,lang_probability,DOUBLE,YES,None,None,None
8,transcript_segments,"STRUCT(""text"" VARCHAR, ""timestamp"" DOUBLE[], words STRUCT(""text"" VARCHAR, ""timestamp"" DOUBLE[])[])[]",YES,None,None,None


In [13]:
row_count = con.execute(
    f"""
    SELECT COUNT(*) AS total_rows
    FROM read_parquet('{PARQUET_PATH.as_posix()}')
    """
).df()

display(row_count)

,total_rows
0,3469
